# **Working with FMV Datasets**

Datamaite includes dataset wrappers for loading, converting, and writing datasets.  All in-memory objects are MAITE-compliant.

**NOTE:** To run this notebook, you must first have a HMIE dataset. You can also download a mock dataset from our [example datasets repository](https://gitlab.jatic.net/jatic/orchestration-interoperability/datamaite-example-datasets/-/tree/main/datasets). 

First, we'll see what dataset formats we have available for loading:

In [ ]:
from datamaite.loaders import available_formats

available_formats()

Now let's load up an **HMIE dataset**

In [ ]:
import os
from pathlib import Path

from datamaite.loaders import load

# Resolve the example datasets. CI sets DATAMAITE_DATASETS_ROOT to the cloned
# `datasets/` directory; locally it falls back to a clone beside this notebook.
# Replace the fallback with your own dataset path if needed.
DATASETS_ROOT = Path(os.environ.get("DATAMAITE_DATASETS_ROOT", "datamaite-example-datasets/datasets"))

hmie_datadir = DATASETS_ROOT / "hmie"

hmie_dataset = load(hmie_datadir, dataset_format="hmie")

type(hmie_dataset)

We can inspect this object by viewing the attributes (`hmie_dataset.metadata`) or by getting the full summary view (`hmie_dataset`).

From here, we can inspect individual entries

In [ ]:
hmie_dataset[0]

In [ ]:
hmie_dataset[0][1].frame_tracks[0]  # hmie_dataset[0][1].frame_tracks[0]

In [ ]:
len(hmie_dataset[0][1].frame_tracks)

In [ ]:
hmie_dataset.metadata

In [ ]:
hmie_dataset.categories

In [ ]:
hmie_dataset.dataset_id

In [ ]:
hmie_dataset.task

In [ ]:
hmie_dataset.num_boxes

In [ ]:
hmie_dataset.sequences[0].boxes[0]  # [0].boxes[0]

## **Dataset conversion**

Let's say we have an existing T&E pipeline that's built on **TAO datasets**. Rather than update the pipeline to test on this one dataset, we can use `datamaite` to **convert this HMIE dataset to TAO**. 

We have two options. First, we'll look at continuing this workflow using the `hmie_dataset` we already have in memory. 

**Note:** `write()`/`convert()` refuse to write into a non-empty destination by default (`mode="error"`). This notebook uses `mode="replace"` below so the cells stay safely re-runnable -- it clears the destination directory before writing.

In [ ]:
from datamaite.writers import write

# Generated datasets are written to an ignored output/ folder beside this notebook.
OUTPUT_DIR = Path("output")
tao_datadir = OUTPUT_DIR / "tao"

# mode="replace" clears tao_datadir first so this cell can be re-run.
write(hmie_dataset, tao_datadir, output_format="tao", mode="replace")

Now we have an on-disk version of the same dataset in TAO format.


The second option for conversions is to **convert direct on disk** without obtaining an in-memory `Dataset` object. 

In [ ]:
from datamaite import convert

tao_datadir2 = OUTPUT_DIR / "tao2"

# mode="replace" clears tao_datadir2 first so this cell can be re-run.
convert(hmie_datadir, tao_datadir2, input_format="hmie", output_format="tao", mode="replace")

We can inspect the output in the browser. 

And just to be sure, we will load the newly created dataset up:

In [ ]:
tao_converted = load(tao_datadir2, dataset_format="tao")

And take a peek at the contents:

In [ ]:
tao_converted.sequences[0]  # .sequences[0]

<div class="alert alert-info">
<b>A note about lossiness when converting INTO fixed taxonomy formats</b>

Converting INTO fixed formats like `MOTChallenge` and `Visdrone Video` format is inherently lossy as the fixed taxonomy dictates the integer categories-to-name relationship. 

For example, an HMIE `"vehicle"` with `category_id=7` comes back as `_VISDRONE_CLASS_NAMES[7] = "tricycle"`. The original label is unrecoverable from the VisDrone files.
</div>

## **Other loaders and conversions...**

A quick overview of the remaining FMV dataset loaders/conversion/writers:


### **MOTChallenge Video format**

In [ ]:
motchallenge_datadir = DATASETS_ROOT / "motchallenge/valid/train"

# convert(hmie_datadir, motchallenge_datadir, input_format="hmie", output_format="motchallenge")

In [ ]:
motchallenge_dataset = load(motchallenge_datadir, dataset_format="motchallenge")

type(motchallenge_dataset)

### **Hugging Face video format**

In [ ]:
# This is an example snippet. A Hugging Face video example will be added to the example datasets repo soon.
# hf_datadir = OUTPUT_DIR / 'huggingface-video'

# convert(hmie_datadir, hf_datadir, input_format="hmie", output_format="huggingface_video_classification")
# hf_dataset = load(path_hf, dataset_format='huggingface_video_classification')
# hf_dataset.metadata

In [ ]:
# hf_dataset.categories

In [ ]:
# hf_dataset.labels

In [ ]:
# hf_dataset.samples

### **Visdrone Video format**

In [ ]:
visdrone_datadir = DATASETS_ROOT / "visdrone/valid/VisDrone2019-VID-train"
visdrone_dataset = load(visdrone_datadir, dataset_format="visdrone_video")

type(visdrone_dataset)

### **TAO format**

In [ ]:
tao_datadir = DATASETS_ROOT / "tao/valid"
tao_dataset = load(tao_datadir, dataset_format="tao")

type(tao_dataset)

### **Flat folder with unlabeled mp4s**

In [ ]:
unlabeled_datadir = DATASETS_ROOT / "flat_mp4/valid"
unlabeled_dataset = load(unlabeled_datadir, dataset_format="flat_mp4")

type(unlabeled_dataset)

<hr>